# StockForecastNet V6 — IT Sector Colab Training

**Before running:**

1. `Runtime → Change runtime type → T4 GPU`
2. Upload files per Section 3 instructions
3. Edit Section 2 (stocks and dates)
4. `Runtime → Run all`


## Section 1 — Setup


In [ ]:
# GPU check
import subprocess, sys, os, pathlib, copy, time, math, random, warnings
import numpy as np
warnings.filterwarnings("ignore")

res = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True)
print("GPU:", res.stdout.strip() if res.returncode==0 else "NOT FOUND — Runtime → T4 GPU")

import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    p = torch.cuda.get_device_properties(0)
    print(f"  {p.name} | {p.total_memory/1024**3:.1f} GB")


In [ ]:
# Install dependencies (yfinance, lightgbm, joblib, pyarrow — rest already in Colab)
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q",
                "yfinance>=0.2.40","lightgbm>=4.0.0","joblib>=1.3.0",
                "pyarrow>=15.0.0","scikit-learn>=1.4.0"], check=True)
import yfinance, lightgbm, joblib, sklearn
print(f"yfinance={yfinance.__version__} lightgbm={lightgbm.__version__} "
      f"joblib={joblib.__version__} sklearn={sklearn.__version__}")


In [ ]:
# Create folder structure and add to Python path
import pathlib, sys

PROJECT_ROOT = pathlib.Path("/content/ai-trading-service")
for d in [PROJECT_ROOT, PROJECT_ROOT/"strategies", PROJECT_ROOT/"utils",
          PROJECT_ROOT/"data", PROJECT_ROOT/"models", PROJECT_ROOT/"exports"]:
    d.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Folders created:")
for d in [PROJECT_ROOT, PROJECT_ROOT/"strategies", PROJECT_ROOT/"utils",
          PROJECT_ROOT/"data", PROJECT_ROOT/"models", PROJECT_ROOT/"exports"]:
    print(f"  {d}")


In [ ]:
# Optional: mount Google Drive to persist models across sessions
USE_DRIVE = False
DRIVE_DIR = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = pathlib.Path("/content/drive/MyDrive/StockForecastNet_V6")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Drive mounted → {DRIVE_DIR}")
else:
    print("Drive not mounted. Download files manually in Section 10.")


## Section 2 — Configuration _(edit this cell)_


In [ ]:
# ── Stocks ────────────────────────────────────────────────────────────────────
SYMBOLS = [
    "TCS.NS", "INFY.NS", "WIPRO.NS", "HCLTECH.NS",
    "TECHM.NS", "LTI.NS", "MPHASIS.NS", "PERSISTENT.NS",
]
START_DATE = "2012-01-01"
END_DATE   = "2025-12-31"

# ── Architecture (V6 defaults for IT-only) ────────────────────────────────────
SEQ_LEN    = 90     # input window days
HORIZON    = 3      # predict N days ahead
PATCH_SIZE = 16     # days per patch
STRIDE     = 8      # patch stride (50% overlap)
D_MODEL    = 96     # embedding dim (64 for <5 stocks, 128 for >10)
N_HEADS    = 4      # must divide D_MODEL
N_LAYERS   = 2
D_FF       = 192    # always 2x D_MODEL
DROPOUT    = 0.2    # higher than V5 (0.1) for correlated IT data

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE      = 128   # reduce to 64 if CUDA OOM
EPOCHS          = 100
LR              = 3e-4
WEIGHT_DECAY    = 1e-3
BCE_WEIGHT      = 0.7   # direction loss (primary)
MSE_WEIGHT      = 0.3   # magnitude loss (regulariser)
PATIENCE        = 30    # MUST be >20 — LR restarts at epoch 20
NOISE_THRESHOLD = 0.001
VAL_SPLIT       = 0.2
GAP             = 10    # days gap between train/val
SEED            = 42

# ── LightGBM ──────────────────────────────────────────────────────────────────
TRAIN_LGBM      = True
LGBM_ESTIMATORS = 500
LGBM_LR         = 0.05

# ── Output files ──────────────────────────────────────────────────────────────
WEIGHTS_FILE = "pretrained_v6.pth"
CONFIG_FILE  = "pretrained_v6_config.pth"
SCALER_FILE  = "scaler_v6.pkl"
LGBM_FILE    = "lgbm_it_model.pkl"
MODELS_DIR   = PROJECT_ROOT / "models"
EXPORTS_DIR  = PROJECT_ROOT / "exports"

n_patches = (SEQ_LEN - PATCH_SIZE) // STRIDE + 1
print(f"Stocks ({len(SYMBOLS)}): {', '.join(SYMBOLS)}")
print(f"Dates: {START_DATE} → {END_DATE}")
print(f"seq={SEQ_LEN} horizon={HORIZON}d patches={n_patches} d_model={D_MODEL}")
print(f"loss = {BCE_WEIGHT}*BCE + {MSE_WEIGHT}*MSE  |  patience={PATIENCE}")


## Section 3 — Upload Project Files

Upload your local files to `/content/ai-trading-service/` using the Colab file
browser (folder icon in left sidebar) or run the upload cell below.

### Files to upload to `/content/ai-trading-service/`

| File             | Source                                   |
| ---------------- | ---------------------------------------- |
| `model_v2.py`    | `apps/ai-trading-service/model_v2.py`    |
| `train_v2.py`    | `apps/ai-trading-service/train_v2.py`    |
| `dataset_v2.py`  | `apps/ai-trading-service/dataset_v2.py`  |
| `features_v2.py` | `apps/ai-trading-service/features_v2.py` |
| `backtest_v2.py` | `apps/ai-trading-service/backtest_v2.py` |
| `lgbm_model.py`  | `apps/ai-trading-service/lgbm_model.py`  |

### Files to upload to `/content/ai-trading-service/utils/`

| File            | Source                                        |
| --------------- | --------------------------------------------- |
| `trading_v2.py` | `apps/ai-trading-service/utils/trading_v2.py` |
| `__init__.py`   | `apps/ai-trading-service/utils/__init__.py`   |

### Files to upload to `/content/ai-trading-service/strategies/`

All 20 strategy files:
`base.py`, `ma_strategy.py`, `macd_strategy.py`, `rsi_strategy.py`,
`bb_strategy.py`, `breakout_strategy.py`, `vwap_strategy.py`, `atr_strategy.py`,
`candlestick_strategy.py`, `momentum_strategy.py`, `stochastic_strategy.py`,
`cci_strategy.py`, `williams_r_strategy.py`, `obv_strategy.py`,
`donchian_strategy.py`, `supertrend_strategy.py`, `keltner_strategy.py`,
`heikin_ashi_strategy.py`, `pivot_strategy.py`, `ichimoku_strategy.py`, `__init__.py`

> **Tip:** Select all files at once with Ctrl+click, then drag into the Colab
> file browser folder.


In [ ]:
# Upload files via dialog (alternative to drag-and-drop)
# Select all .py files at once — cell auto-places them in the right folders
from google.colab import files as colab_files
import shutil

STRATEGY_NAMES = {
    "base.py","ma_strategy.py","macd_strategy.py","rsi_strategy.py",
    "bb_strategy.py","breakout_strategy.py","vwap_strategy.py","atr_strategy.py",
    "candlestick_strategy.py","momentum_strategy.py","stochastic_strategy.py",
    "cci_strategy.py","williams_r_strategy.py","obv_strategy.py",
    "donchian_strategy.py","supertrend_strategy.py","keltner_strategy.py",
    "heikin_ashi_strategy.py","pivot_strategy.py","ichimoku_strategy.py",
}
UTIL_NAMES = {"trading_v2.py"}
# __init__.py is ambiguous — upload strategies/__init__.py and utils/__init__.py separately

uploaded = colab_files.upload()
for fname in uploaded:
    if fname in STRATEGY_NAMES:
        dest = PROJECT_ROOT / "strategies" / fname
    elif fname in UTIL_NAMES:
        dest = PROJECT_ROOT / "utils" / fname
    else:
        dest = PROJECT_ROOT / fname
    shutil.move(fname, str(dest))
    print(f"  {fname} → {dest}")
print("Done. Run next cell to verify.")


In [ ]:
# Verify all required files exist, then import everything
import sys
for m in ["model_v2","features_v2","dataset_v2","backtest_v2","lgbm_model",
          "utils","utils.trading_v2","strategies","strategies.base"]:
    sys.modules.pop(m, None)

required = [
    PROJECT_ROOT/"model_v2.py", PROJECT_ROOT/"features_v2.py",
    PROJECT_ROOT/"dataset_v2.py", PROJECT_ROOT/"lgbm_model.py",
    PROJECT_ROOT/"utils"/"trading_v2.py",
    PROJECT_ROOT/"strategies"/"base.py",
    PROJECT_ROOT/"strategies"/"ichimoku_strategy.py",
]
missing = [str(f) for f in required if not f.exists()]
if missing:
    print("MISSING — upload these files before continuing:")
    for m in missing: print(f"  {m}")
    raise FileNotFoundError("Upload files first")

from model_v2    import StockForecastNet
from features_v6 import add_features_v6, FEATURE_COLS
from dataset_v2  import StockDatasetV2, extract_time_features
from utils.trading_v2 import generate_signal_v2, CONFIDENCE_FLOOR
from lgbm_model  import LGBMDirectionModel, build_tree_features

# Quick smoke test to verify V6 forward signature
_m = StockForecastNet(n_features=56, seq_len=SEQ_LEN, horizon=HORIZON,
                      patch_size=PATCH_SIZE, stride=STRIDE, d_model=D_MODEL,
                      n_heads=N_HEADS, n_layers=N_LAYERS, d_ff=D_FF)
import torch
_x, _tf = torch.randn(2,SEQ_LEN,56), torch.randn(2,SEQ_LEN,6)
_logit, _mag, _stats = _m(_x, _tf)
assert _logit.shape==(2,) and _mag.shape==(2,HORIZON), "V6 forward signature mismatch"
c = _m.count_parameters()
del _m, _x, _tf, _logit, _mag, _stats

print("All imports OK | V6 smoke test passed")
print(f"  V6 forward: (logit, mag_norm, revin_stats) — no denorm in training path")
print(f"  Total params: {c['total']:,} ({c['size_mb']} MB)")
print(f"  FEATURE_COLS: {len(FEATURE_COLS)} | CONFIDENCE_FLOOR: {CONFIDENCE_FLOOR}")


## Section 4 — Fetch NSE Data


In [ ]:
# Fetch daily OHLCV from Yahoo Finance (free, no API key)
import yfinance as yf
import pandas as pd

def fetch_stock(ticker, start, end):
    print(f"  {ticker:<25}", end="", flush=True)
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
    if df.empty: print("FAILED"); return None
    df.columns = [c[0].lower() if isinstance(df.columns, pd.MultiIndex) else c.lower()
                  for c in df.columns]
    df = df.reset_index().rename(columns={"date":"datetime","Date":"datetime"})
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df[["datetime","open","high","low","close","volume"]].dropna()
    df = df.sort_values("datetime").reset_index(drop=True)
    print(f"  {len(df):,} rows [{df['datetime'].min().date()} → {df['datetime'].max().date()}]")
    return df

print(f"Fetching {len(SYMBOLS)} IT stocks | {START_DATE} → {END_DATE}")
RAW_DATA = {}
for ticker in SYMBOLS:
    df = fetch_stock(ticker, START_DATE, END_DATE)
    if df is not None and len(df) >= 500:
        RAW_DATA[ticker] = df
        sym = ticker.replace(".NS","")
        df.to_parquet(PROJECT_ROOT/"data"/f"{sym}_daily.parquet", index=False)

print(f"\n{len(RAW_DATA)} stocks ready: {list(RAW_DATA.keys())}")


In [ ]:
# Alternative: upload your own Upstox parquet files
UPLOAD_PARQUET = False
if UPLOAD_PARQUET:
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    for fname, content in uploaded.items():
        sym = fname.split("_")[0].upper()
        p = PROJECT_ROOT/"data"/fname
        p.write_bytes(content)
        df = pd.read_parquet(p)
        df.columns = [c.lower() for c in df.columns]
        if "datetime" not in df.columns:
            df = df.reset_index().rename(columns={"date":"datetime"})
        df["datetime"] = pd.to_datetime(df["datetime"])
        df = df[["datetime","open","high","low","close","volume"]].dropna()
        RAW_DATA[sym+".NS"] = df.sort_values("datetime").reset_index(drop=True)
        print(f"  {fname}: {len(df):,} rows")


## Section 5 — Feature Engineering and Datasets


In [ ]:
# Compute 56 stationary technical features for each stock (all 19 strategies)
FEATURED_DATA = {}
for ticker, df_raw in RAW_DATA.items():
    print(f"  {ticker:<25}", end="", flush=True)
    try:
        df_f = add_features_v6(df_raw.copy())
        n, nr = len(df_f), len(df_raw)
        df_f = df_f.copy()
        df_f["close"]    = df_raw["close"].values[nr-n : nr]
        df_f["datetime"] = df_raw["datetime"].values[nr-n : nr]
        FEATURED_DATA[ticker] = df_f
        print(f"  {n:,} rows ({nr-n} warmup dropped)")
    except Exception as e:
        print(f"  FAILED: {e}")

print(f"\n{len(FEATURED_DATA)} stocks | {len(FEATURE_COLS)} features each")


In [ ]:
# Chronological 80/20 train/val split per stock + shared RobustScaler
from sklearn.preprocessing import RobustScaler
from torch.utils.data import ConcatDataset
import joblib

train_dfs, val_dfs = {}, {}
for ticker, df in FEATURED_DATA.items():
    n = len(df); nv = int(n*VAL_SPLIT); nt = n - nv - GAP
    if nt < SEQ_LEN + HORIZON + 100: print(f"  SKIP {ticker}: {nt} rows"); continue
    train_dfs[ticker] = df.iloc[:nt].copy()
    val_dfs[ticker]   = df.iloc[nt+GAP:].copy()
    if "datetime" in df.columns:
        print(f"  {ticker:<25}: {nt:,} train [{df['datetime'].iloc[0].date()}→"
              f"{df['datetime'].iloc[nt-1].date()}]")

print("\nFitting shared RobustScaler on all training data...")
combined = np.vstack([df[FEATURE_COLS].values for df in train_dfs.values()])
SHARED_SCALER = RobustScaler().fit(combined)
print(f"  Fitted on {len(combined):,} rows from {len(train_dfs)} stocks")

TRAIN_SETS, VAL_SETS = [], []
for ticker, df in train_dfs.items():
    try:
        ds = StockDatasetV2(df, window=SEQ_LEN, horizon=HORIZON,
                            noise_threshold=NOISE_THRESHOLD,
                            scaler=SHARED_SCALER, symbol=ticker)
        ds.summary(); TRAIN_SETS.append(ds)
    except ValueError as e: print(f"  SKIP {ticker}: {e}")
for ticker, df in val_dfs.items():
    try:
        ds = StockDatasetV2(df, window=SEQ_LEN, horizon=HORIZON,
                            noise_threshold=NOISE_THRESHOLD,
                            scaler=SHARED_SCALER, symbol=ticker)
        VAL_SETS.append(ds)
    except ValueError as e: print(f"  SKIP val {ticker}: {e}")

TRAIN_DS = ConcatDataset(TRAIN_SETS) if len(TRAIN_SETS)>1 else TRAIN_SETS[0]
VAL_DS   = ConcatDataset(VAL_SETS)   if len(VAL_SETS)>1   else VAL_SETS[0]
n_tr = sum(len(d) for d in TRAIN_SETS); n_va = sum(len(d) for d in VAL_SETS)
eff  = n_tr * len(FEATURE_COLS) / 168000
print(f"\n{n_tr:,} train + {n_va:,} val | eff_ratio={eff:.1f}x",
      "LOW" if eff<2 else "MARGINAL" if eff<5 else "GOOD")


## Section 6 — Build Model and Train


In [ ]:
# Build V6 model on GPU
import torch, copy

torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

MODEL = StockForecastNet(
    n_features=len(FEATURE_COLS), seq_len=SEQ_LEN, horizon=HORIZON,
    patch_size=PATCH_SIZE, stride=STRIDE, d_model=D_MODEL,
    n_heads=N_HEADS, n_layers=N_LAYERS, d_ff=D_FF, dropout=DROPOUT,
).to(DEVICE)
print(MODEL)
c = MODEL.count_parameters()
print(f"Params: {c['total']:,} ({c['size_mb']} MB)")


In [ ]:
# Balanced batch iterator (50/50 UP/DOWN per batch — no DataLoader/multinomial crash)
import torch.nn as nn

def _balanced_batches(dataset, bs, shuffle, device):
    if isinstance(dataset, ConcatDataset):
        labels = torch.cat([d._primary_labels for d in dataset.datasets])
    else:
        labels = dataset._primary_labels
    up_idx   = (labels>0).nonzero(as_tuple=True)[0].tolist()
    down_idx = (labels<=0).nonzero(as_tuple=True)[0].tolist()
    if shuffle: random.shuffle(up_idx); random.shuffle(down_idx)
    half = bs//2
    for i in range(min(len(up_idx),len(down_idx))//half):
        batch = up_idx[i*half:(i+1)*half] + down_idx[i*half:(i+1)*half]
        if shuffle: random.shuffle(batch)
        Xs,tfs,ys = [],[],[]
        for idx in batch:
            x,tf,y = dataset[idx]; Xs.append(x); tfs.append(tf); ys.append(y)
        yield torch.stack(Xs).to(device), torch.stack(tfs).to(device), torch.stack(ys).to(device)

def _n_batches(ds, bs):
    if isinstance(ds, ConcatDataset):
        lbl = torch.cat([d._primary_labels for d in ds.datasets])
    else: lbl = ds._primary_labels
    return min(int((lbl>0).sum()),int((lbl<=0).sum())) // (bs//2)

def _norm_labels(y, stats):
    _, std = stats
    return y / (std[:,0,0].unsqueeze(-1) + 1e-8)

def train_v6(model, train_ds, val_ds, device, batch_size=BATCH_SIZE, epochs=EPOCHS,
             lr=LR, patience=PATIENCE, horizon=HORIZON, weight_decay=WEIGHT_DECAY,
             bce_w=BCE_WEIGHT, mse_w=MSE_WEIGHT):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=20, T_mult=2, eta_min=1e-6)
    bce_fn = nn.BCEWithLogitsLoss(); mse_fn = nn.MSELoss()
    n_tr = _n_batches(train_ds,batch_size); n_va = _n_batches(val_ds,batch_size)
    best_loss,best_acc,best_state,no_imp = float("inf"),0.,None,0

    print("  Dry-run...", end=" ")
    model.eval()
    with torch.no_grad():
        x0,tf0,y0 = train_ds[0]
        l0,m0,_ = model(x0.unsqueeze(0).to(device), tf0.unsqueeze(0).to(device))
    print(f"OK logit={tuple(l0.shape)} mag={tuple(m0.shape)}")
    del x0,tf0,y0,l0,m0

    print(f"  {'Ep':>4}  {'TrLoss':>9}  {'TrAcc':>6}  {'VaLoss':>9}  {'VaAcc':>6}  {'LR':>9}  {'s/ep':>6}")
    print("  " + "-"*64)

    for ep in range(1, epochs+1):
        t0=time.time(); model.train(); tl=tc=tt=0
        for X,tf,y in _balanced_batches(train_ds, batch_size, True, device):
            logit,mag,stats = model(X,tf)
            dl = (y[:,-1]>0).float(); yn = _norm_labels(y,stats)
            loss = bce_w*bce_fn(logit,dl) + mse_w*mse_fn(mag,yn)
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
            tl+=loss.item(); pd_=(torch.sigmoid(logit)>=0.5).float()
            tc+=int((pd_==dl).sum()); tt+=y.size(0)
        sch.step(); model.eval(); vl=vc=vt=0
        with torch.no_grad():
            for X,tf,y in _balanced_batches(val_ds, batch_size, False, device):
                logit,mag,stats = model(X,tf); dl=(y[:,-1]>0).float(); yn=_norm_labels(y,stats)
                vl+=(bce_w*bce_fn(logit,dl)+mse_w*mse_fn(mag,yn)).item()
                pd_=(torch.sigmoid(logit)>=0.5).float(); vc+=int((pd_==dl).sum()); vt+=y.size(0)
        ta=tc/max(tt,1); va=vc/max(vt,1); atr=tl/max(n_tr,1); ava=vl/max(n_va,1)
        lr_=opt.param_groups[0]["lr"]; el=time.time()-t0
        print(f"  {ep:>4}  {atr:>9.5f}  {ta:>6.3f}  {ava:>9.5f}  {va:>6.3f}  {lr_:>9.2e}  {el:>6.1f}s", flush=True)
        if ep==1: print(f"  [ETA ~{el*epochs/60:.0f} min — early stop usually at 60-85 ep]")
        if ava < best_loss:
            best_loss,best_acc,no_imp = ava,va,0
            best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            no_imp+=1
            if no_imp>=patience:
                print(f"\n  Early stop ep {ep}. Best val acc: {best_acc:.2%}"); break
    return best_state, best_acc

print("Training function ready")


In [ ]:
# Run training — epoch 1 VaAcc should be ~50% (V5 bug showed 47.7% due to denorm bias)
print("="*55, "\n  TRAINING StockForecastNet V6")
print(f"  Device: {DEVICE} | {sum(len(d) for d in TRAIN_SETS):,} train | {sum(len(d) for d in VAL_SETS):,} val")
print(f"  Loss: {BCE_WEIGHT}*BCE + {MSE_WEIGHT}*MSE (no denorm in loss path)")
print("="*55)

t0 = time.time()
BEST_STATE, BEST_ACC = train_v6(
    MODEL, TRAIN_DS, VAL_DS, DEVICE,
    batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR, patience=PATIENCE,
    horizon=HORIZON, weight_decay=WEIGHT_DECAY, bce_w=BCE_WEIGHT, mse_w=MSE_WEIGHT)
mins = (time.time()-t0)/60

print(f"\nDone in {mins:.1f} min | Best val acc: {BEST_ACC:.2%}")
print("STRONG" if BEST_ACC>=0.58 else "GOOD" if BEST_ACC>=0.54 else "MARGINAL" if BEST_ACC>=0.52 else "WEAK")


## Section 7 — LightGBM Training


In [ ]:
# Train LightGBM direction classifier on ~580 tabular features
# Expected: trains in 2-5 min, often matches or beats transformer on IT-only data
if not TRAIN_LGBM:
    print("Skipped (TRAIN_LGBM=False)"); LGBM_MODEL=None; LGBM_METRICS={}
else:
    LGBM_MODEL = LGBMDirectionModel(
        horizon=HORIZON, n_estimators=LGBM_ESTIMATORS, learning_rate=LGBM_LR,
        max_depth=6, num_leaves=63, min_child_samples=50,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, seed=SEED)
    t0 = time.time()
    LGBM_METRICS = LGBM_MODEL.fit(
        [(t,df) for t,df in train_dfs.items()],
        [(t,df) for t,df in val_dfs.items()])
    print(f"\nLightGBM done in {(time.time()-t0)/60:.1f} min")
    for k,v in LGBM_METRICS.items(): print(f"  {k:<25}: {v:.2%}")

    print("\nTop 20 features (which indicators predict IT direction):")
    for _,row in LGBM_MODEL.top_features(n=20).iterrows():
        print(f"  {row['feature']:<38} {row['pct']:>5.1f}%  {'|'*max(1,int(row['pct']/1.5))}")


## Section 8 — Evaluation


In [ ]:
# Load best weights; create CPU copy (deepcopy avoids in-place device mutation)
if BEST_STATE: MODEL.load_state_dict(BEST_STATE)
MODEL_CPU = copy.deepcopy(MODEL).cpu().eval()
print(f"MODEL: {next(MODEL.parameters()).device} | MODEL_CPU: cpu (independent copy)")


In [ ]:
# Compare Transformer vs LightGBM
import pandas as pd
print("="*50, "\n  MODEL COMPARISON")
print(f"  {'Transformer V6':<25} {BEST_ACC:>7.2%}")
if LGBM_METRICS:
    lgbm_va = LGBM_METRICS.get("val_acc",0)
    print(f"  {'LightGBM':<25} {lgbm_va:>7.2%}")
    diff = lgbm_va - BEST_ACC
    if diff > 0.02:   rec = f"LightGBM better → use lgbm_weight=0.65"
    elif diff < -0.02: rec = f"Transformer better → use lgbm_weight=0.35"
    else:              rec = f"Similar → use default lgbm_weight=0.55"
    print(f"  Recommendation: {rec}")
print("="*50)


In [ ]:
# Per-stock accuracy breakdown
print("Per-stock val accuracy:")
for vds in VAL_SETS:
    n=len(vds); correct=strong_c=strong_n=0
    for i in range(n):
        x,tf,y = vds[i]
        with torch.no_grad():
            logit,_,_ = MODEL_CPU(x.unsqueeze(0), tf.unsqueeze(0))
        p_up = float(torch.sigmoid(logit[0]).item())
        pred = 1 if p_up>=0.5 else 0; conf = p_up if pred==1 else 1-p_up
        if pred==(1 if y[-1].item()>0 else 0): correct+=1
        if conf>=0.70: strong_n+=1; strong_c += (pred==(1 if y[-1].item()>0 else 0))
    acc=correct/n; sacc=strong_c/strong_n if strong_n else 0
    bar="="*int(acc*20)+"-"*(20-int(acc*20))
    print(f"  {vds.symbol:<22} [{bar}] {acc:.2%}  strong {strong_c}/{strong_n}={sacc:.0%}")


In [ ]:
# Feature attention weights — which indicators V6 focused on
all_attn = np.zeros(len(FEATURE_COLS)); ns=0
for vds in VAL_SETS:
    for i in range(min(len(vds),300)):
        x,tf,_ = vds[i]
        with torch.no_grad():
            _,_,_,attn_w = MODEL_CPU.forward(x.unsqueeze(0), tf.unsqueeze(0), return_attn_weights=True)
        all_attn += attn_w[0].numpy(); ns+=1
avg_attn = all_attn/ns
feat_df = pd.DataFrame({"feature":FEATURE_COLS,"pct":avg_attn/avg_attn.sum()*100}).sort_values("pct",ascending=False)
print("Top 20 by V6 attention (compare with LightGBM importance — agreement = reliable predictor):")
for _,row in feat_df.head(20).iterrows():
    print(f"  {row['feature']:<32} {row['pct']:>5.1f}%  {'|'*max(1,int(row['pct']*2))}")


In [ ]:
# Signal distribution on validation set
from collections import Counter
sig_counts = Counter()
for vds in VAL_SETS:
    for i in range(len(vds)):
        x,tf,y = vds[i]
        with torch.no_grad():
            logit,mag,stats = MODEL_CPU(x.unsqueeze(0),tf.unsqueeze(0))
        p_up = float(torch.sigmoid(logit[0]).item())
        conf = p_up if p_up>=0.5 else 1-p_up
        mag_d = MODEL_CPU.revin.denormalize(mag[0], stats)
        sig,sth = generate_signal_v2(1 if p_up>=0.5 else 0, conf, float(mag_d[-1].item()))
        sig_counts[f"{sig} {sth}" if sig!="HOLD" else "HOLD"] += 1
n_total = sum(sig_counts.values())
print("Signal distribution (validation set):")
for k,v in sorted(sig_counts.items()):
    print(f"  {k:<15} {v:>6,}  ({v/n_total:.1%})")
trade_rate = (n_total - sig_counts.get("HOLD",0))/n_total
print(f"\nTrade rate: {trade_rate:.1%}  (if >60%, raise CONFIDENCE_FLOOR in utils/trading_v2.py)")


## Section 9 — Backtest


In [ ]:
# Full portfolio backtest (real share quantities, brokerage+slippage, long-only)
from backtest_v2 import backtest_v2

BACKTEST_TICKER = list(FEATURED_DATA.keys())[0]   # change as needed

bt_ds = StockDatasetV2(FEATURED_DATA[BACKTEST_TICKER], window=SEQ_LEN, horizon=HORIZON,
                        noise_threshold=0.0, scaler=SHARED_SCALER, symbol=BACKTEST_TICKER)
bt_ds.summary()

BT_RESULTS = backtest_v2(
    model=MODEL_CPU, dataset=bt_ds, horizon=HORIZON,
    min_confidence=CONFIDENCE_FLOOR, position_size_pct=0.20,
    device="cpu", log_trades=True, log_interval=500,
    csv_path=str(EXPORTS_DIR/"backtest_trades.csv"))


In [ ]:
# Multi-stock backtest summary
print("="*60, "\n  MULTI-STOCK BACKTEST SUMMARY", "\n"+"="*60)
print(f"  {'Stock':<22} {'Return':>8} {'WinRate':>8} {'Trades':>7} {'Sharpe':>8}")
print("  "+"-"*55)
for ticker in list(FEATURED_DATA.keys()):
    try:
        ds = StockDatasetV2(FEATURED_DATA[ticker], window=SEQ_LEN, horizon=HORIZON,
                            noise_threshold=0.0, scaler=SHARED_SCALER, symbol=ticker)
        r = backtest_v2(model=MODEL_CPU, dataset=ds, horizon=HORIZON,
                        min_confidence=CONFIDENCE_FLOOR, position_size_pct=0.20,
                        device="cpu", log_trades=False, log_interval=0)
        m = "+" if r["total_return_pct"]>0 else "-"
        print(f"  {m} {ticker:<20} {r['total_return_pct']:>+7.1f}% {r['accuracy']*100:>7.1f}%"
              f" {r['n_trades']:>7} {r['sharpe_ratio']:>8.2f}")
    except Exception as e: print(f"  - {ticker:<20} ERROR: {e}")
print("\nTip: if Sharpe is negative → raise CONFIDENCE_FLOOR to 0.65 in utils/trading_v2.py")


## Section 10 — Save and Download Artifacts


In [ ]:
# Save transformer weights, config, and scaler
import joblib, torch

if BEST_STATE: MODEL.load_state_dict(BEST_STATE)
MODELS_DIR.mkdir(exist_ok=True); EXPORTS_DIR.mkdir(exist_ok=True)

w_path = MODELS_DIR / WEIGHTS_FILE
c_path = MODELS_DIR / CONFIG_FILE
s_path = MODELS_DIR / SCALER_FILE
l_path = MODELS_DIR / LGBM_FILE

torch.save(BEST_STATE or MODEL.state_dict(), w_path)
torch.save(MODEL.get_config(), c_path)
joblib.dump(SHARED_SCALER, s_path)
print(f"  {w_path.name:<35} {w_path.stat().st_size/1024:>6.1f} KB")
print(f"  {c_path.name:<35} {c_path.stat().st_size/1024:>6.1f} KB")
print(f"  {s_path.name:<35} {s_path.stat().st_size/1024:>6.1f} KB")

if LGBM_MODEL is not None:
    LGBM_MODEL.save(str(l_path))
    print(f"  {l_path.name:<35} {l_path.stat().st_size/1024:>6.1f} KB")

# Verify reload
cfg_v = torch.load(c_path, map_location="cpu")
m_v = StockForecastNet(**{**cfg_v,"dropout":0.0})
m_v.load_state_dict(torch.load(w_path, map_location="cpu"), strict=False); m_v.eval()
with torch.no_grad():
    _l,_m,_ = m_v(torch.randn(1,SEQ_LEN,len(FEATURE_COLS)), torch.randn(1,SEQ_LEN,6))
assert _l.shape==(1,) and _m.shape==(1,HORIZON)
print(f"Reload OK | logit={tuple(_l.shape)} mag={tuple(_m.shape)}")
del m_v,_l,_m

if DRIVE_DIR:
    import shutil
    for f in [w_path,c_path,s_path] + ([l_path] if LGBM_MODEL else []):
        shutil.copy(f, DRIVE_DIR/f.name); print(f"  Drive: {DRIVE_DIR/f.name}")


In [ ]:
# Download to your computer
from google.colab import files as colab_files

dl = [w_path, c_path, s_path]
if LGBM_MODEL is not None: dl.append(l_path)

for fpath in dl:
    print(f"  Downloading {fpath.name}...", end=" ")
    colab_files.download(str(fpath)); print("done"); time.sleep(0.5)

print("\nCopy to apps/ai-trading-service/:")
for fp in dl: print(f"  {fp.name}")
print("\nThen run:")
print("  python infer.py --symbol TCS")
print("  python infer.py --symbol TCS --model ensemble")
print("  uvicorn api_v2:app --host 0.0.0.0 --port 8000")


## Section 11 — Fine-tune on Single Stock (Optional)


In [ ]:
FINETUNE        = False
FINETUNE_TICKER = "TCS.NS"

if not FINETUNE:
    print("Skipped. Set FINETUNE=True to run.")
elif FINETUNE_TICKER not in FEATURED_DATA:
    print(f"ERROR: {FINETUNE_TICKER} not in FEATURED_DATA")
else:
    df_ft = FEATURED_DATA[FINETUNE_TICKER]; n=len(df_ft)
    nv=int(n*VAL_SPLIT); nt=n-nv-GAP
    ft_tr = StockDatasetV2(df_ft.iloc[:nt], window=SEQ_LEN, horizon=HORIZON,
                           noise_threshold=NOISE_THRESHOLD, scaler=SHARED_SCALER, symbol=FINETUNE_TICKER)
    ft_va = StockDatasetV2(df_ft.iloc[nt+GAP:], window=SEQ_LEN, horizon=HORIZON,
                           noise_threshold=NOISE_THRESHOLD, scaler=SHARED_SCALER, symbol=FINETUNE_TICKER)
    ft_tr.summary(); ft_va.summary()

    # Must explicitly call .to(DEVICE) — .to() is in-place, MODEL may have drifted to cpu
    MODEL.load_state_dict(BEST_STATE); MODEL.to(DEVICE)
    print(f"Model on: {next(MODEL.parameters()).device} | LR: {LR/5:.2e}")

    ft_best, ft_acc = train_v6(MODEL, ft_tr, ft_va, DEVICE,
        batch_size=BATCH_SIZE, epochs=40, lr=LR/5, patience=20,
        horizon=HORIZON, weight_decay=WEIGHT_DECAY, bce_w=BCE_WEIGHT, mse_w=MSE_WEIGHT)

    print(f"\nFine-tune: {ft_acc:.2%}  |  Pretrain: {BEST_ACC:.2%}")
    if ft_acc > BEST_ACC + 0.003:
        BEST_STATE = ft_best; BEST_ACC = ft_acc
        MODEL.load_state_dict(BEST_STATE)
        MODEL_CPU = copy.deepcopy(MODEL).cpu().eval()
        print("Fine-tuned weights adopted. MODEL_CPU updated.")
    else:
        MODEL.load_state_dict(BEST_STATE); MODEL.to(DEVICE)
        MODEL_CPU = copy.deepcopy(MODEL).cpu().eval()
        print("Pretrained weights kept (fine-tune did not improve sufficiently).")


## Section 12 — Quick Inference Test


In [ ]:
# Test V6 inference on most recent data — replicates python infer.py --output json
TEST_TICKER = list(FEATURED_DATA.keys())[0]
df_inf = FEATURED_DATA[TEST_TICKER]; nr = len(df_inf)

X_t  = torch.tensor(SHARED_SCALER.transform(df_inf.tail(SEQ_LEN)[FEATURE_COLS].values),
                    dtype=torch.float32).unsqueeze(0)
tf_t = torch.tensor(extract_time_features(df_inf, nr-SEQ_LEN, SEQ_LEN),
                    dtype=torch.float32).unsqueeze(0)

MODEL_CPU.eval()
with torch.no_grad():
    logit,mag,stats,attn_w = MODEL_CPU.forward(X_t, tf_t, return_attn_weights=True)

p_up  = float(torch.sigmoid(logit[0]).item())
d     = 1 if p_up>=0.5 else 0
conf  = p_up if d==1 else 1-p_up
mag_d = MODEL_CPU.revin.denormalize(mag[0], stats)
pred  = float(mag_d[-1].item())
steps = [round(float(v),4) for v in mag_d.tolist()]
agree = all(s>0 for s in steps) or all(s<0 for s in steps)
sig,sth = generate_signal_v2(d, conf, pred)
attn = attn_w[0].tolist()
top5 = {FEATURE_COLS[i]:round(attn[i],4) for i in sorted(range(len(attn)),key=lambda i:attn[i],reverse=True)[:5]}
latest = df_inf["datetime"].iloc[-1].date() if "datetime" in df_inf.columns else "?"

print(f"INFERENCE — {TEST_TICKER}  (latest: {latest})")
print(f"  Signal:     {sig}  ({sth})")
print(f"  Direction:  {'UP' if d==1 else 'DOWN'}  (p_up={p_up:.1%}  conf={conf:.1%})")
print(f"  Primary {HORIZON}d: {pred:+.2%}")
print(f"  All steps:  {[f'{s:+.2%}' for s in steps]}")
print(f"  Agreement:  {'Yes' if agree else 'No'}")
print(f"  Top features: {top5}")

if LGBM_MODEL:
    lr_ = LGBM_MODEL.predict_latest(df_inf)
    print(f"\nLightGBM: {lr_['direction_label']} p_up={lr_['p_up']:.1%} conf={lr_['confidence']:.1%}")
    ens = 0.55*lr_["p_up"] + 0.45*p_up
    print(f"Ensemble:  {'UP' if ens>=0.5 else 'DOWN'} p_up={ens:.1%}")
